# Spherical Fidelity Plots

This notebook generates the two polar plots from the manuscript showing which metric
operators pass or fail the adversarial threshold, visualised as points on a sphere
parameterised by $(r, \\theta, \\phi)$:

- **Vertical plane**: meridional slice with $\\phi = -\\pi/2$ and $\\phi = \\pi/2$,
  showing how the threshold outcome varies with $\\theta$.
- **Horizontal plane**: equatorial slice at $\\theta = \\pi/2$, showing how the
  threshold outcome varies with $\\phi$.

Pass/fail results are read from `data/processed/experimental_fidelities_master.csv`.
Figures are saved to `figures/` as PDF and PNG.

Run the notebook from the repository root.

In [ ]:
import os
from math import pi

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Embed fonts as TrueType in the PDF output
matplotlib.rcParams['pdf.fonttype'] = 42

## Parameters

In [ ]:
# Select which experiment to plot: 'qutrit' or 'qubit'
which_experiment = 'qutrit'

INPUT_PATH = "data/processed/experimental_fidelities_master.csv"
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Metric operator parameter grid

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Load pass/fail results and prepare plot data

In [ ]:
grid = np.array(eta_grid())[:, :3]  # columns: r, theta, phi

# Load pass/fail column for the selected experiment
df = pd.read_csv(INPUT_PATH)
passed_col = 'qutrit_passed' if which_experiment == 'qutrit' else 'qubit_passed'
passed = df[passed_col].to_numpy(dtype=bool)

panel = '(b)' if which_experiment == 'qubit' else '(a)'
exp   = 'Two-qubit' if which_experiment == 'qubit' else 'Qutrit'

# Horizontal plane: the 20 metrics with theta = pi/2. The polar angle of the plot is phi.
in_horizontal = np.isclose(grid[:, 1], pi / 2)
h_r      = grid[in_horizontal, 0]
h_angle  = grid[in_horizontal, 2]
h_passed = passed[in_horizontal]

# Vertical plane: the 20 metrics with phi = -pi/2 or +pi/2. The polar angle of the plot
# is measured from the top; phi = -pi/2 is drawn on the left half (counter-clockwise)
# and phi = +pi/2 on the right half (clockwise).
in_vertical = ~in_horizontal
v_r      = grid[in_vertical, 0]
v_angle  = np.where(grid[in_vertical, 2] < 0, grid[in_vertical, 1], -grid[in_vertical, 1])
v_passed = passed[in_vertical]


def scatter_pass_fail(ax, angle, r, ok):
    ax.scatter(angle[ok],  r[ok],  c='c', label='Above threshold')
    ax.scatter(angle[~ok], r[~ok], c='m', label='Below threshold')


def save_figure(fname):
    plt.savefig(fname + '.pdf', bbox_inches='tight')
    plt.savefig(fname + '.png', bbox_inches='tight')
    print(f"Saved to {fname}.pdf / .png")

## Vertical plane plot

Meridional slice with $\\phi = -\\pi/2$ (left) and $\\phi = \\pi/2$ (right), showing
threshold outcomes as $\\theta$ varies.

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))

scatter_pass_fail(ax, v_angle, v_r, v_passed)

ax.set_theta_offset(np.pi / 2)
ax.set_xticks(np.arange(0, 2*np.pi, np.pi/4))
ax.set_xticklabels([
    r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$',
    r'$\pi$', r'$3\pi/4$', r'$\pi/2$', r'$\pi/4$'
])

ax.set_title(
    fr"{panel}  {exp} Vertical Plane (varying $\theta$)" + "\n"
    + r"Left: $\phi=-\pi/2$  |  Right: $\phi=\pi/2$",
    loc='center', fontweight='bold', pad=15
)
ax.plot([0.5, 0.5], [0, 1], color='black', linestyle='--',
        transform=ax.transAxes, clip_on=False)
ax.grid(True)
ax.set_rmax(1)
ax.set_rticks([0.2, 0.4, 0.6, 0.8])
ax.set_rlabel_position(80)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False, fontsize=11)

plt.tight_layout()
save_figure(os.path.join(OUTPUT_DIR, f"{which_experiment}_vertical"))
plt.show()

## Horizontal plane plot

Equatorial slice at $\\theta = \\pi/2$, showing threshold outcomes as $\\phi$ varies.

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))

scatter_pass_fail(ax, h_angle, h_r, h_passed)

ax.set_theta_offset(np.pi / 2)
ax.set_xticks(np.arange(0, 2*np.pi, np.pi/4))
ax.set_xticklabels([
    r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$',
    r'$\pi$', r'$5\pi/4$', r'$3\pi/2$', r'$7\pi/4$'
])

ax.set_title(
    fr"{panel}  {exp} Horizontal Plane (varying $\phi$)" + "\n"
    + r"$\theta=\pi/2$",
    loc='center', fontweight='bold', pad=15
)
ax.grid(True)
ax.set_rmax(1)
ax.set_rticks([0.2, 0.4, 0.6, 0.8])
ax.set_rlabel_position(80)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=2, frameon=False, fontsize=11)

plt.tight_layout()
save_figure(os.path.join(OUTPUT_DIR, f"{which_experiment}_horizontal"))
plt.show()